# LIAR Benchmark / LIAR 基准

ISOT 和 WELFake 的标签都是按 **网站** 打的，模型只要认出来源就能答对（`04`–`06`）。LIAR（Wang, 2017, ACL）的 12,836 条政治陈述由 PolitiFact 编辑 **逐条** 核查：
- 同一个政治人物既有真话也有假话，"认出是谁说的"不足以判断真假；
- 所有样本都是同一种格式（一句话），没有"路透社文风 vs 博客文风"可以利用。

所以 LIAR 上的分数代表 **"只看文字判断真假"的真实难度**。

**只使用陈述文本**：LIAR 附带说话人、党派和"历史核查次数"（第 9–13 列）。按数据集说明，历史核查次数 *包含当前这条陈述本身*，用作特征就是标签泄漏；说话人和党派则是另一种"认来源"的捷径。代码见 `src/liar.py`。

ISOT and WELFake are labelled per **site**; LIAR's 12,836 political statements are fact-checked **one by one** by PolitiFact, with the same speakers saying both true and false things, all in one format. Scores here measure how hard truthfulness detection from text really is. Only the statement text is used: the credit-history columns include the current label (leakage) and speaker / party are another source shortcut.

## 1. Setup and data / 环境与数据

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.analysis import paired_bootstrap_auc_diff
from src.external import LIAR_BINARY, load_liar
from src.liar import LIAR_C_GRID, LIAR_MAX_LENGTH

RESULTS = Path('../results/liar')
pd.set_option('display.width', 200)

liar = load_liar().set_index('doc_id')
print(liar.groupby('split')['label'].agg(n='size', real_share='mean').round(3))
print('\nMedian statement length (words):', int(liar['text'].str.split().str.len().median()))

           n  real_share
split                   
test    1283       0.567
train  10269       0.562
val     1284       0.520

Median statement length (words): 17


In [2]:
# 六级标签及其二分类映射 / The six labels and their binary mapping
order = ['pants-fire', 'false', 'barely-true', 'half-true', 'mostly-true', 'true']
pd.DataFrame({'count': liar['label6'].value_counts().reindex(order),
              'binary label (1 = real)': pd.Series(LIAR_BINARY).reindex(order)})

,count,binary label (1 = real)
pants-fire,1050,0
false,2511,0
barely-true,2108,0
half-true,2638,1
mostly-true,2466,1
true,2063,1


二分类采用文献中最常见的划分：true / mostly-true / half-true → 真，barely-true / false / pants-fire → 假。**half-true 本身含糊**，放在哪一边都有争议；原始的六级标签保留在 `label6` 中。

The common binary split is used (half-true → real); half-true is inherently ambiguous, and the original labels are kept in `label6`.

## 2. Models / 模型

| 模型 | 训练数据 | 设置 |
|---|---|---|
| TF-IDF + 逻辑回归 | LIAR train | C 在 LIAR val 上选择；范围向下扩展到 0.004（LIAR 很小，需要更强的正则化） |
| DistilBERT（3 个种子） | LIAR train | notebook 03 的设置，max_length 改为 64（覆盖 99.95% 的陈述），最佳 epoch 在 LIAR val 上选择 |
| ISOT 的 TF-IDF 模型 | ISOT train | 直接用在 LIAR 上；长文章 vs 一句话，格式差别很大，**只作参考** |

In [3]:
rec = json.loads((RESULTS / 'tfidf_liar.json').read_text())
print('LIAR C grid:', LIAR_C_GRID, '| chosen C =', rec['C'], '| DistilBERT max_length =', LIAR_MAX_LENGTH)
pd.DataFrame(rec['C_table']).set_index('C')

LIAR C grid: (0.004, 0.016, 0.0625, 0.25, 1.0, 4.0) | chosen C = 0.25 | DistilBERT max_length = 64


,val accuracy %,val AUC
C,,
0.0040,52.0249,0.6708
0.0160,52.0249,0.6722
0.0625,54.8287,0.6766
0.2500,62.1495,0.6810
1.0000,62.3053,0.6769
4.0000,62.6947,0.6608


## 3. Results / 结果

In [4]:
def load(name):
    return json.loads((RESULTS / f'{name}.json').read_text())

cols = ['majority baseline %', 'accuracy %', 'balanced accuracy %', 'AUC', 'predicted real %']
rows = {'ISOT TF-IDF model (reference only)': load('tfidf_isot')['metrics']['LIAR test'],
        'TF-IDF trained on LIAR': load('tfidf_liar')['metrics']['LIAR test']}
for seed in (42, 43, 44):
    rows[f'DistilBERT trained on LIAR, seed {seed}'] = load(f'distilbert_seed{seed}')['metrics']['LIAR test']
per_model = pd.DataFrame(rows).T[cols]
per_model

,majority baseline %,accuracy %,balanced accuracy %,AUC,predicted real %
ISOT TF-IDF model (reference only),56.7,47.8,51.4,0.538,23.1
TF-IDF trained on LIAR,56.7,62.5,58.9,0.672,78.6
"DistilBERT trained on LIAR, seed 42",56.7,64.4,62.0,0.684,69.5
"DistilBERT trained on LIAR, seed 43",56.7,65.0,63.8,0.687,61.1
"DistilBERT trained on LIAR, seed 44",56.7,63.5,62.2,0.675,61.7


In [5]:
bert = per_model.loc[[i for i in per_model.index if i.startswith('DistilBERT')], ['accuracy %', 'balanced accuracy %', 'AUC']]
pd.DataFrame({'mean': bert.astype(float).mean().round(3), 'std': bert.astype(float).std().round(3)}).T

,accuracy %,balanced accuracy %,AUC
mean,64.300,62.667,0.682
std,0.755,0.987,0.006


In [6]:
# DistilBERT 的训练过程：验证 loss 在第 1–2 个 epoch 后就开始上升（很快开始过拟合）
# DistilBERT's validation loss starts rising after epoch 1–2 (quick overfitting)
pd.DataFrame({f'seed {s}': {'best epoch': load(f'distilbert_seed{s}')['info']['best_epoch'],
                            'val loss per epoch': load(f'distilbert_seed{s}')['info']['history']['val_loss']}
              for s in (42, 43, 44)}).T

,best epoch,val loss per epoch
seed 42,2,"[0.66765, 0.65988, 0.71238]"
seed 43,2,"[0.65131, 0.64576, 0.71327]"
seed 44,1,"[0.64953, 0.68519, 0.72643]"


## 4. Is DistilBERT better than TF-IDF? / DistilBERT 比 TF-IDF 好吗

配对 bootstrap：在同一批 LIAR test 陈述上，比较两个模型的 AUC。

Paired bootstrap of the AUC difference on the same LIAR test statements.

In [7]:
def probs(name, split='LIAR test'):
    d = np.load(RESULTS / f'{name}.npz', allow_pickle=True)
    return pd.Series(d[f'prob::{split}'], index=d[f'doc_id::{split}'].astype(str))

tfidf = probs('tfidf_liar')
y = liar.loc[tfidf.index, 'label']
rows = {}
for seed in (42, 43, 44):
    r = paired_bootstrap_auc_diff(y, probs(f'distilbert_seed{seed}').loc[tfidf.index], tfidf)
    rows[f'seed {seed}'] = {'AUC(DistilBERT) - AUC(TF-IDF)': r['AUC difference (A - B)'], '95% CI': r['95% CI']}
pd.DataFrame(rows).T

,AUC(DistilBERT) - AUC(TF-IDF),95% CI
seed 42,0.012,"(-0.01, 0.033)"
seed 43,0.015,"(-0.007, 0.038)"
seed 44,0.004,"(-0.017, 0.024)"


## 5. Does the model follow the six truth levels? / 模型的判断是否随真假等级变化

如果模型学到了一些判断真假的能力，从 pants-fire 到 true，被判为"真"的比例应该 **逐级升高**。

If the model has learned anything about truthfulness, the share predicted real should rise from pants-fire to true.

In [8]:
label6 = liar.loc[tfidf.index, 'label6']
bert_mean = pd.concat([probs(f'distilbert_seed{s}').loc[tfidf.index] for s in (42, 43, 44)], axis=1).mean(axis=1)
pd.DataFrame({
    'n': label6.value_counts().reindex(order),
    'TF-IDF: predicted real %': ((tfidf > 0.5).groupby(label6).mean() * 100).reindex(order).round(1),
    'DistilBERT (3-seed mean prob): predicted real %': ((bert_mean > 0.5).groupby(label6).mean() * 100).reindex(order).round(1),
    'DistilBERT: mean P(real)': bert_mean.groupby(label6).mean().reindex(order).round(3),
})

,n,TF-IDF: predicted real %,DistilBERT (3-seed mean prob): predicted real %,DistilBERT: mean P(real)
label6,,,,
pants-fire,92,54.3,34.8,0.470
false,250,71.6,50.0,0.530
barely-true,214,71.0,56.5,0.546
half-true,267,81.3,67.8,0.597
mostly-true,249,89.6,83.5,0.659
true,211,88.6,82.0,0.658


## 6. The real difficulty / 真实难度

把每个模型在 ISOT 内部的成绩，和在 LIAR 上的成绩放在一起（ISOT 数字来自 `02`、`03`、`05`）。

Each model's in-domain ISOT score next to its LIAR score (ISOT numbers from `02`, `03`, `05`).

In [9]:
from src.experiments import load_results

isot = load_results()
isot = isot[isot['eval set'] == 'ISOT test'].groupby(['model', 'config'])[['accuracy %', 'AUC']].mean()
pd.DataFrame({
    'ISOT test accuracy %': [isot.loc[('tfidf', 'C64'), 'accuracy %'], isot.loc[('distilbert', 'DistilBERT'), 'accuracy %']],
    'ISOT test AUC': [isot.loc[('tfidf', 'C64'), 'AUC'], isot.loc[('distilbert', 'DistilBERT'), 'AUC']],
    'LIAR test accuracy %': [per_model.loc['TF-IDF trained on LIAR', 'accuracy %'], bert['accuracy %'].astype(float).mean()],
    'LIAR test AUC': [per_model.loc['TF-IDF trained on LIAR', 'AUC'], bert['AUC'].astype(float).mean()],
}, index=['TF-IDF', 'DistilBERT (3-seed mean)']).round(3)

,ISOT test accuracy %,ISOT test AUC,LIAR test accuracy %,LIAR test AUC
TF-IDF,98.700,0.999,62.5,0.672
DistilBERT (3-seed mean),98.967,0.999,64.3,0.682


## 7. Conclusions / 结论

**1. 去掉来源捷径后，只看文字判断真假非常困难。**

| | ISOT test（标签 = 来源） | LIAR test（逐条核查） |
|---|---|---|
| TF-IDF | 98.7%，AUC 0.999 | **62.5%**，AUC **0.672** |
| DistilBERT（3 个种子） | 99.0%，AUC 0.999 | **64.3 ± 0.8%**，AUC **0.682 ± 0.006** |
| 全部猜真 | 55.0% | 56.7% |

在 LIAR 上，两个模型都只比"全部猜真"高 6–8 个百分点。**ISOT 上的 99% 与这里约 64% 之间约 35 个百分点的差距，就是"认出来源"这个捷径的分量。**

**2. 在 LIAR 上，DistilBERT 没有可靠地超过 TF-IDF。**
AUC 只高 0.004–0.015，3 个种子的置信区间都包含 0。DistilBERT 在第 1–2 个 epoch 后验证 loss 就开始上升，很快过拟合。一句话是真是假，往往取决于外部事实（例如"去年失业率下降了 2%"），文本本身提供不了这些信息，预训练的语言知识也补不上。

**3. 模型确实学到了一点真实的信号，但很弱。**
从 pants-fire 到 mostly-true，被判为"真"的比例逐级升高（DistilBERT：34.8% → 83.5%；TF-IDF：54.3% → 89.6%）。但 DistilBERT 给出的平均"真新闻概率"只从 0.47 升到 0.66，两端的区分并不大。

**4. ISOT 模型在 LIAR 上接近随机**（AUC 0.538），并且把 77% 的陈述判为假，因为它们"不像路透社"。由于格式差别很大，这一项只作参考。

**对项目的意义**：只看文字的模型，在去掉来源捷径之后，AUC 只能达到约 0.68。要做得更好，判断必须依赖文本之外的信息，也就是第 5 步的 **证据核查**。LIAR 就是检验它的公平场地：核对证据的方法应该能明显超过 0.68。

**局限**：half-true 归为"真"有争议；LIAR 是一句话的陈述，不是整篇新闻；这里只使用了陈述文本。

---

1. **Without the source shortcut, truthfulness from text is hard**: TF-IDF 62.5% / AUC 0.672 and DistilBERT 64.3 ± 0.8% / AUC 0.682 on LIAR, versus ~99% on ISOT — the ~35-point gap is what the source shortcut was worth.
2. **DistilBERT does not reliably beat TF-IDF on LIAR** (+0.004 to +0.015 AUC, all CIs include 0) and overfits after 1–2 epochs: truth often depends on outside facts the text does not contain.
3. **There is a weak genuine signal**: the share predicted real rises from pants-fire to mostly-true (DistilBERT 34.8% → 83.5%), but the mean P(real) only moves from 0.47 to 0.66.
4. **The ISOT model is near chance on LIAR** (AUC 0.538; reference only).

**Implication**: text-only models top out around AUC 0.68 once the shortcut is gone; doing better needs information beyond the text — evidence-based checking (step 5), with LIAR as the test bed. **Limitations**: half-true → real is debatable; statements are not articles; only the statement text is used.